# Advanced Problems: Lambdas and Sorting

This notebook contains advanced practice problems with complete solutions. The focus is on Python's `sorted`, `list.sort`, `key` functions, lambda expressions, tuple keys, stable sorting, dictionaries, complex objects, and robust sorting of messy data.

**Best-practice reminders**

- Prefer a named function when the key logic is reused or non-trivial.
- Prefer built-ins such as `str.casefold`, `abs`, `len`, `operator.itemgetter`, and `operator.attrgetter` when they express the idea clearly.
- Use tuple keys for multi-level sorting.
- Remember that Python sorting is stable: equal keys keep their original relative order.
- Do not use `cmp`-style comparison unless absolutely necessary; key functions are usually clearer and faster.

## Setup

In [1]:
from operator import itemgetter, attrgetter
from dataclasses import dataclass
from pprint import pprint
import math
import re

## Problem 1 — Case-insensitive sorting with deterministic tie-breaking

You are given a list of words with mixed case and accents. Sort them alphabetically in a case-insensitive way using `casefold`, but when two words compare equal after case-folding, place the original all-lowercase version before the mixed/uppercase version.

Return a new sorted list. Do not mutate the original list.

**Input**

In [2]:
words = ['Banana', 'apple', 'Äpfel', 'banana', 'Apple', 'äpfel', 'cherry', 'Cherry']
words

['Banana', 'apple', 'Äpfel', 'banana', 'Apple', 'äpfel', 'cherry', 'Cherry']

### Solution

In [3]:
sorted_words = sorted(words, key=lambda s: (s.casefold(), not s.islower(), s))
sorted_words

['apple', 'Apple', 'banana', 'Banana', 'cherry', 'Cherry', 'äpfel', 'Äpfel']

### Explanation

The key is a tuple:

1. `s.casefold()` performs stronger case-insensitive comparison than `lower()`.
2. `not s.islower()` is `False` for lowercase strings and `True` otherwise, so lowercase comes first.
3. `s` is a final deterministic tie-breaker.

This is a reasonable use of a lambda because the key is short and used once.

## Problem 2 — Sort dictionary keys by value, then by key

Given a dictionary of product names and quantities, return the product names sorted by:

1. Quantity descending.
2. Product name ascending, case-insensitive.

**Input**

In [4]:
inventory = {
    'notebook': 12,
    'Pencil': 50,
    'eraser': 50,
    'Marker': 12,
    'binder': 7,
    'pen': 50,
}
inventory

{'notebook': 12,
 'Pencil': 50,
 'eraser': 50,
 'Marker': 12,
 'binder': 7,
 'pen': 50}

### Solution

In [5]:
sorted_products = sorted(inventory, key=lambda name: (-inventory[name], name.casefold()))
sorted_products

['eraser', 'pen', 'Pencil', 'Marker', 'notebook', 'binder']

### Explanation

`sorted(inventory)` iterates over dictionary keys. The key function returns `(-quantity, normalized_name)`. Negating the quantity gives descending numeric order while keeping the whole sort ascending.

## Problem 3 — Sort records with missing values

You are given student records. Some students have missing scores represented by `None`. Sort students by:

1. Missing scores last.
2. Score descending.
3. Name ascending, case-insensitive.

**Input**

In [6]:
students = [
    {'name': 'Maya', 'score': 91},
    {'name': 'liam', 'score': None},
    {'name': 'Ava', 'score': 91},
    {'name': 'Noah', 'score': 84},
    {'name': 'zoe', 'score': None},
    {'name': 'Emma', 'score': 100},
]
students

[{'name': 'Maya', 'score': 91},
 {'name': 'liam', 'score': None},
 {'name': 'Ava', 'score': 91},
 {'name': 'Noah', 'score': 84},
 {'name': 'zoe', 'score': None},
 {'name': 'Emma', 'score': 100}]

### Solution

In [7]:
def student_sort_key(student):
    score = student['score']
    return (
        score is None,                  # False comes before True, so real scores first
        0 if score is None else -score, # descending score for non-missing values
        student['name'].casefold(),
    )

sorted_students = sorted(students, key=student_sort_key)
pprint(sorted_students)

[{'name': 'Emma', 'score': 100},
 {'name': 'Ava', 'score': 91},
 {'name': 'Maya', 'score': 91},
 {'name': 'Noah', 'score': 84},
 {'name': 'liam', 'score': None},
 {'name': 'zoe', 'score': None}]


### Explanation

This is better as a named function because the logic is multi-step and benefits from comments. The first tuple item separates valid scores from missing scores. The second sorts valid scores descending. The third gives a stable alphabetical tie-breaker.

## Problem 4 — Sort complex numbers by geometric distance, then angle

Python does not define a default ordering for complex numbers. Sort complex numbers by:

1. Distance from the origin.
2. Polar angle in radians.

Use squared distance rather than actual distance to avoid unnecessary square roots.

**Input**

In [8]:
points = [3+4j, 1+1j, -1+1j, 0+0j, 2+0j, -3-4j, 1-1j]
points

[(3+4j), (1+1j), (-1+1j), 0j, (2+0j), (-3-4j), (1-1j)]

### Solution

In [9]:
def complex_key(z):
    squared_distance = z.real ** 2 + z.imag ** 2
    angle = math.atan2(z.imag, z.real)
    return (squared_distance, angle)

sorted_points = sorted(points, key=complex_key)
sorted_points

[0j, (1-1j), (1+1j), (-1+1j), (2+0j), (-3-4j), (3+4j)]

### Explanation

A key function lets us sort values that are not naturally orderable. Squared distance preserves the same distance ordering as real distance but avoids `sqrt`, making it cheaper and simpler.

## Problem 5 — Stable sorting: preserve original priority order

A task list is already arranged by original priority from most important to least important. You now want to group tasks by status in this order: `blocked`, `in_progress`, `todo`, `done`.

Within each status group, preserve the original order.

**Input**

In [10]:
tasks = [
    {'id': 101, 'status': 'todo', 'title': 'write tests'},
    {'id': 102, 'status': 'blocked', 'title': 'deploy API'},
    {'id': 103, 'status': 'todo', 'title': 'refactor parser'},
    {'id': 104, 'status': 'in_progress', 'title': 'update docs'},
    {'id': 105, 'status': 'blocked', 'title': 'fix auth'},
    {'id': 106, 'status': 'done', 'title': 'create ticket'},
]
tasks

[{'id': 101, 'status': 'todo', 'title': 'write tests'},
 {'id': 102, 'status': 'blocked', 'title': 'deploy API'},
 {'id': 103, 'status': 'todo', 'title': 'refactor parser'},
 {'id': 104, 'status': 'in_progress', 'title': 'update docs'},
 {'id': 105, 'status': 'blocked', 'title': 'fix auth'},
 {'id': 106, 'status': 'done', 'title': 'create ticket'}]

### Solution

In [11]:
status_order = {'blocked': 0, 'in_progress': 1, 'todo': 2, 'done': 3}

sorted_tasks = sorted(tasks, key=lambda task: status_order[task['status']])
pprint(sorted_tasks)

[{'id': 102, 'status': 'blocked', 'title': 'deploy API'},
 {'id': 105, 'status': 'blocked', 'title': 'fix auth'},
 {'id': 104, 'status': 'in_progress', 'title': 'update docs'},
 {'id': 101, 'status': 'todo', 'title': 'write tests'},
 {'id': 103, 'status': 'todo', 'title': 'refactor parser'},
 {'id': 106, 'status': 'done', 'title': 'create ticket'}]


### Explanation

Python's sort is stable. Because the key only sorts by status, tasks with the same status keep their original relative order. This is useful when the original order already contains meaningful priority.

## Problem 6 — Multi-pass stable sort vs tuple-key sort

You have employee records. Sort by department ascending, salary descending, and name ascending.

Solve it two ways:

1. With one tuple key.
2. With multiple stable sorts.

**Input**

In [12]:
employees = [
    {'name': 'Iris', 'department': 'Engineering', 'salary': 130_000},
    {'name': 'Omar', 'department': 'Sales', 'salary': 95_000},
    {'name': 'Chen', 'department': 'Engineering', 'salary': 130_000},
    {'name': 'Nina', 'department': 'Sales', 'salary': 110_000},
    {'name': 'Luis', 'department': 'Engineering', 'salary': 115_000},
]
employees

[{'name': 'Iris', 'department': 'Engineering', 'salary': 130000},
 {'name': 'Omar', 'department': 'Sales', 'salary': 95000},
 {'name': 'Chen', 'department': 'Engineering', 'salary': 130000},
 {'name': 'Nina', 'department': 'Sales', 'salary': 110000},
 {'name': 'Luis', 'department': 'Engineering', 'salary': 115000}]

### Solution A — tuple key

In [13]:
by_tuple_key = sorted(
    employees,
    key=lambda e: (e['department'].casefold(), -e['salary'], e['name'].casefold())
)
pprint(by_tuple_key)

[{'department': 'Engineering', 'name': 'Chen', 'salary': 130000},
 {'department': 'Engineering', 'name': 'Iris', 'salary': 130000},
 {'department': 'Engineering', 'name': 'Luis', 'salary': 115000},
 {'department': 'Sales', 'name': 'Nina', 'salary': 110000},
 {'department': 'Sales', 'name': 'Omar', 'salary': 95000}]


### Solution B — multiple stable sorts

In [14]:
by_stable_passes = employees.copy()
by_stable_passes.sort(key=lambda e: e['name'].casefold())
by_stable_passes.sort(key=lambda e: e['salary'], reverse=True)
by_stable_passes.sort(key=lambda e: e['department'].casefold())

pprint(by_stable_passes)

[{'department': 'Engineering', 'name': 'Chen', 'salary': 130000},
 {'department': 'Engineering', 'name': 'Iris', 'salary': 130000},
 {'department': 'Engineering', 'name': 'Luis', 'salary': 115000},
 {'department': 'Sales', 'name': 'Nina', 'salary': 110000},
 {'department': 'Sales', 'name': 'Omar', 'salary': 95000}]


### Check that both methods agree

In [15]:
by_tuple_key == by_stable_passes

True

### Explanation

For multi-pass sorting, sort by the least important criterion first and the most important criterion last. Stability preserves the earlier ordering inside later equal-key groups. The tuple-key version is often more compact; multi-pass sorting can be clearer when each stage is conceptually separate.

## Problem 7 — Sorting dataclass instances with `attrgetter`

Create a list of `Book` objects. Sort them by:

1. Author ascending.
2. Year ascending.
3. Title ascending.

Use `operator.attrgetter` when possible.

**Input**

In [16]:
@dataclass(frozen=True)
class Book:
    title: str
    author: str
    year: int

books = [
    Book('The Dispossessed', 'Ursula K. Le Guin', 1974),
    Book('Foundation', 'Isaac Asimov', 1951),
    Book('I, Robot', 'Isaac Asimov', 1950),
    Book('A Wizard of Earthsea', 'Ursula K. Le Guin', 1968),
]
books

[Book(title='The Dispossessed', author='Ursula K. Le Guin', year=1974),
 Book(title='Foundation', author='Isaac Asimov', year=1951),
 Book(title='I, Robot', author='Isaac Asimov', year=1950),
 Book(title='A Wizard of Earthsea', author='Ursula K. Le Guin', year=1968)]

### Solution

In [17]:
sorted_books = sorted(books, key=attrgetter('author', 'year', 'title'))
sorted_books

[Book(title='I, Robot', author='Isaac Asimov', year=1950),
 Book(title='Foundation', author='Isaac Asimov', year=1951),
 Book(title='A Wizard of Earthsea', author='Ursula K. Le Guin', year=1968),
 Book(title='The Dispossessed', author='Ursula K. Le Guin', year=1974)]

### Explanation

`attrgetter('author', 'year', 'title')` is clearer than `lambda b: (b.author, b.year, b.title)` when you are simply reading attributes. A lambda is better when you need transformations such as `casefold()` or computed values.

## Problem 8 — Natural sorting of filenames

Default string sorting gives surprising results for filenames like `file10.txt` and `file2.txt`. Implement a natural-sort key so numeric parts are compared as integers.

**Input**

In [18]:
filenames = ['file10.txt', 'file2.txt', 'file1.txt', 'file20.txt', 'file11.txt', 'file3.txt']
filenames

['file10.txt',
 'file2.txt',
 'file1.txt',
 'file20.txt',
 'file11.txt',
 'file3.txt']

### Solution

In [19]:
number_pattern = re.compile(r'(\d+)')

def natural_key(text):
    parts = number_pattern.split(text)
    return [int(part) if part.isdigit() else part.casefold() for part in parts]

sorted_filenames = sorted(filenames, key=natural_key)
sorted_filenames

['file1.txt',
 'file2.txt',
 'file3.txt',
 'file10.txt',
 'file11.txt',
 'file20.txt']

### Explanation

The key converts `'file10.txt'` into something like `['file', 10, '.txt']`, so `10` is compared as a number instead of as the character `'1'` followed by `'0'`.

## Problem 9 — Sorting by expensive keys: prove the key is called once per item

Python's sort computes the key once for each item, stores it internally, and then sorts using those key values.

Create a key function that counts how often it is called. Sort a list and verify that the count equals the number of items.

**Input**

In [20]:
values = ['pear', 'apple', 'fig', 'banana', 'kiwi']
values

['pear', 'apple', 'fig', 'banana', 'kiwi']

### Solution

In [21]:
calls = 0

def counting_key(s):
    global calls
    calls += 1
    return (len(s), s)

sorted_values = sorted(values, key=counting_key)

sorted_values, calls, len(values)

(['fig', 'kiwi', 'pear', 'apple', 'banana'], 5, 5)

### Explanation

The key function was called exactly once per item. This is why key-based sorting is usually preferred over repeated comparison functions for expensive transformations.

## Problem 10 — Rank leaderboard entries with mixed criteria

You have leaderboard rows with username, score, completion time, and submission order. Sort by:

1. Score descending.
2. Completion time ascending.
3. Original submission order ascending.

Return only the usernames in rank order.

**Input**

In [22]:
leaderboard = [
    {'user': 'neo', 'score': 900, 'time_seconds': 320, 'submitted_at': 3},
    {'user': 'trinity', 'score': 950, 'time_seconds': 410, 'submitted_at': 1},
    {'user': 'morpheus', 'score': 950, 'time_seconds': 390, 'submitted_at': 2},
    {'user': 'oracle', 'score': 900, 'time_seconds': 300, 'submitted_at': 4},
    {'user': 'smith', 'score': 950, 'time_seconds': 390, 'submitted_at': 0},
]
leaderboard

[{'user': 'neo', 'score': 900, 'time_seconds': 320, 'submitted_at': 3},
 {'user': 'trinity', 'score': 950, 'time_seconds': 410, 'submitted_at': 1},
 {'user': 'morpheus', 'score': 950, 'time_seconds': 390, 'submitted_at': 2},
 {'user': 'oracle', 'score': 900, 'time_seconds': 300, 'submitted_at': 4},
 {'user': 'smith', 'score': 950, 'time_seconds': 390, 'submitted_at': 0}]

### Solution

In [23]:
ranked = sorted(
    leaderboard,
    key=lambda row: (-row['score'], row['time_seconds'], row['submitted_at'])
)

ranked_users = [row['user'] for row in ranked]
ranked_users

['smith', 'morpheus', 'trinity', 'oracle', 'neo']

### Explanation

A tuple key cleanly expresses mixed ascending and descending criteria. For descending numeric fields, negate the value. For ascending fields, use the value directly.

## Challenge Problem — Sort heterogeneous values safely

You are given a list containing integers, floats, strings, `None`, and complex numbers. Python cannot directly sort these mixed types.

Sort by type group in this order:

1. Real numbers: `int` and `float`, sorted numerically.
2. Strings, sorted case-insensitively.
3. Complex numbers, sorted by distance from origin.
4. `None` values last.

**Input**

In [24]:
mixed = [3, 'Banana', None, 2.5, 1+1j, 'apple', -4, 3+4j, 'Cherry', None, 0]
mixed

[3, 'Banana', None, 2.5, (1+1j), 'apple', -4, (3+4j), 'Cherry', None, 0]

### Solution

In [25]:
def mixed_key(value):
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return (0, value)
    if isinstance(value, str):
        return (1, value.casefold())
    if isinstance(value, complex):
        return (2, value.real ** 2 + value.imag ** 2)
    if value is None:
        return (3, 0)
    raise TypeError(f'Unsupported value: {value!r}')

sorted_mixed = sorted(mixed, key=mixed_key)
sorted_mixed

[-4, 0, 2.5, 3, 'apple', 'Banana', 'Cherry', (1+1j), (3+4j), None, None]

### Explanation

The first item in the key tuple is a type-rank. This ensures values of different types are never compared directly. The second item gives the ordering inside each group.

## Summary

You practiced:

- Sorting with lambdas and named key functions.
- Sorting dictionaries by values.
- Sorting records using tuple keys.
- Handling missing data.
- Sorting non-orderable types such as complex numbers.
- Using stable sorting intentionally.
- Choosing `itemgetter`/`attrgetter` when they make code clearer.
- Building robust keys for natural sorting and heterogeneous data.